In [27]:
import hashlib
import json
import os
import sqlite3
import sys
import urllib.error
import urllib.request
import zipfile

HF_REPO = "cbdb/cbdb-sqlite"
HF_LATEST_URL = "https://huggingface.co/datasets/%s/resolve/main/latest.zip" % HF_REPO
HF_META_URL = "https://huggingface.co/datasets/%s/resolve/main/latest.json" % HF_REPO

DOWNLOAD_DIR = "db"
SOURCE = "huggingface"
DB_PATH = None
OUT_JSON = "surnames_raw.json"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
print("配置完成：下载到", DOWNLOAD_DIR, "| 来源", SOURCE)
print("下载地址:", HF_LATEST_URL)

配置完成：下载到 db | 来源 huggingface
下载地址: https://huggingface.co/datasets/cbdb/cbdb-sqlite/resolve/main/latest.zip


In [28]:
import hashlib
import json
import os
import sys
import urllib.error
import urllib.request
import zipfile

HF_REPO = "cbdb/cbdb-sqlite"
HF_LATEST_URL = "https://huggingface.co/datasets/%s/resolve/main/latest.zip" % HF_REPO
HF_META_URL = "https://huggingface.co/datasets/%s/resolve/main/latest.json" % HF_REPO
DATAVERSE_DOI = "doi:10.7910/DVN/PAGGQS"
DATAVERSE_URL = "https://dataverse.harvard.edu/api/access/dataset/:persistentId?persistentId=" + DATAVERSE_DOI


def _stream(url, dest, retries=3):
    tmp = dest + ".part"
    for attempt in range(1, retries + 1):
        try:
            done = 0
            mode = "wb"
            req = urllib.request.Request(url, headers={"User-Agent": "cbdb-downloader/1.0"})
            if os.path.exists(tmp) and os.path.getsize(tmp) > 0:
                done = os.path.getsize(tmp)
                req.add_header("Range", "bytes=%d-" % done)
                mode = "ab"
            with urllib.request.urlopen(req, timeout=60) as resp:
                total = int(resp.headers.get("Content-Length") or 0) + done
                with open(tmp, mode) as f:
                    while True:
                        chunk = resp.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
                        done += len(chunk)
                        if total:
                            sys.stdout.write("\r  %8.1f / %8.1f MB (%5.1f%%)" % (done / 1e6, total / 1e6, done * 100 / total))
                            sys.stdout.flush()
            os.replace(tmp, dest)
            print("\n  OK:", dest)
            return dest
        except (urllib.error.URLError, OSError) as e:
            print("\n  %d/%d failed: %s" % (attempt, retries, e))
            if attempt == retries:
                return None
    return None


def _fetch_json(url):
    req = urllib.request.Request(url, headers={"User-Agent": "cbdb-downloader/1.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)


def download_huggingface(outdir):
    print("[A] 读取元数据:", HF_META_URL)
    meta = _fetch_json(HF_META_URL)
    print("     包内数据库:", meta.get("sqlite_filename"), "| 生成时间(UTC):", meta.get("generated_at_utc"))
    print("[B] 下载 latest.zip:", HF_LATEST_URL, "(约 139 MB)")
    downloaded = _stream(HF_LATEST_URL, os.path.join(outdir, "latest.zip"))
    if not downloaded:
        return None
    sha = meta.get("sha256")
    if sha:
        h = hashlib.sha256()
        with open(downloaded, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        actual = h.hexdigest()
        if actual == sha:
            print("  sha256 校验通过:", actual[:16] + "...")
        else:
            print("  !! sha256 不匹配：期望 %s，实际 %s" % (sha, actual))
    return downloaded


def download_dataverse(outdir):
    print("[C] Dataverse 官方镜像:", DATAVERSE_URL)
    return _stream(DATAVERSE_URL, os.path.join(outdir, "CBDB_Dataverse.zip"))


def find_sqlite(root):
    for dirpath, _, files in os.walk(root):
        for fn in files:
            if fn.lower().endswith((".sqlite", ".sqlite3")):
                return os.path.join(dirpath, fn)
    return None


def resolve_db(downloaded, outdir):
    if not downloaded:
        return None
    if downloaded.lower().endswith(".zip"):
        print("解压 ZIP:", downloaded)
        with open(downloaded, "rb") as f:
            magic = f.read(2)
        if magic != b"PK":
            print("  !! 文件头不是 ZIP(PK)，放弃使用")
            return None
        with zipfile.ZipFile(downloaded) as z:
            z.extractall(outdir)
        db = find_sqlite(outdir)
        if db:
            print("db found:", db)
            return db
        accdb = [os.path.join(dp, fn) for dp, _, fs in os.walk(outdir) for fn in fs if fn.lower().endswith(".accdb")]
        print("ZIP 内没有 .sqlite；找到 .accdb:", accdb)
        return None
    return downloaded


def get_db(primary, outdir):
    if primary == "local":
        return find_sqlite(outdir)
    if primary == "huggingface":
        db = resolve_db(download_huggingface(outdir), outdir)
        if db:
            return db
        print("HuggingFace 下载/解压失败。")
        return None
    return resolve_db(download_dataverse(outdir), outdir)

print("下载函数已定义")

下载函数已定义


In [29]:
DB_PATH = get_db(SOURCE, DOWNLOAD_DIR)
if DB_PATH is None:
    raise SystemExit("未找到 .sqlite：请检查网络。")

print("数据库:", DB_PATH)

[A] 读取元数据: https://huggingface.co/datasets/cbdb/cbdb-sqlite/resolve/main/latest.json
     包内数据库: cbdb_20260808.sqlite3 | 生成时间(UTC): 2026-08-08T19:14:20Z
[B] 下载 latest.zip: https://huggingface.co/datasets/cbdb/cbdb-sqlite/resolve/main/latest.zip (约 139 MB)
     138.7 /    138.7 MB (100.0%)
  OK: db/latest.zip
  !! sha256 不匹配：期望 7dccd5cffceb14bef7895c3d6cd2fe7971374226d8a256e565a872c798cacc55，实际 a4bf47f7fa05a887d44659872c037e0304143668c36132ff60f50cc28a58f588
解压 ZIP: db/latest.zip
db found: db/cbdb_20260808.sqlite3
数据库: db/cbdb_20260808.sqlite3


In [30]:
con = sqlite3.connect("file:" + DB_PATH + "?mode=ro", uri=True)
con.execute("PRAGMA temp_store=MEMORY")

# 确认 BIOG_MAIN 存在，列出列名
cols = con.execute("PRAGMA table_info(BIOG_MAIN)").fetchall()
print("BIOG_MAIN 列数:", len(cols))
print("前 20 列:", [c[1] for c in cols][:20])

# 看姓氏字段（Top 10）
print("\n姓氏 Top 10 示例：")
for row in con.execute(
    "SELECT c_surname_chn, COUNT(*) FROM BIOG_MAIN "
    "WHERE c_surname_chn IS NOT NULL AND c_surname_chn <> '' "
    "GROUP BY c_surname_chn ORDER BY 2 DESC LIMIT 10"
):
    print("  ", row)
con.close()

BIOG_MAIN 列数: 55
前 20 列: ['c_personid', 'c_name', 'c_name_chn', 'c_index_year', 'c_index_year_type_code', 'c_index_year_source_id', 'c_female', 'c_index_addr_id', 'c_index_addr_type_code', 'c_ethnicity_code', 'c_household_status_code', 'c_tribe', 'c_birthyear', 'c_by_nh_code', 'c_by_nh_year', 'c_by_range', 'c_deathyear', 'c_dy_nh_code', 'c_dy_nh_year', 'c_dy_range']

姓氏 Top 10 示例：
   ('李', 41344)
   ('王', 39023)
   ('張', 35457)
   ('陳', 26376)
   ('劉', 23937)
   ('趙', 16548)
   ('楊', 15916)
   ('吳', 14728)
   ('黃', 12650)
   ('周', 11796)


In [31]:
SQL_SURNAME_COUNT = """
SELECT c_surname_chn, COUNT(*)
FROM BIOG_MAIN
WHERE c_surname_chn IS NOT NULL
  AND c_surname_chn <> ''
GROUP BY c_surname_chn
"""


def extract_surnames(db_path, out_path=None):
    con = sqlite3.connect("file:" + db_path + "?mode=ro", uri=True)
    con.execute("PRAGMA temp_store=MEMORY")
    rows = con.execute(SQL_SURNAME_COUNT).fetchall()
    con.close()
    data = {}
    for surname, count in rows:
        data[surname] = data.get(surname, 0) + count
    if out_path:
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=1)
    return data

print("SQL 常量已定义 :", "SQL_SURNAME_COUNT")
print("提取函数已定义 :", "extract_surnames(db_path, out_path=None)")

SQL 常量已定义 : SQL_SURNAME_COUNT
提取函数已定义 : extract_surnames(db_path, out_path=None)


In [32]:
surnames = extract_surnames(DB_PATH, OUT_JSON)
print("不同姓氏字符串数:", len(surnames))
print("有姓人物总数    :", sum(surnames.values()))
print("已保存          :", OUT_JSON)

不同姓氏字符串数: 2874
有姓人物总数    : 647915
已保存          : surnames_raw.json


In [33]:
print("Top 20 原始计数（未查重）")
for i, (s, n) in enumerate(sorted(surnames.items(), key=lambda x: -x[1])[:20], 1):
    print("%2d. %s  %s" % (i, s, n))

Top 20 原始计数（未查重）
 1. 李  41344
 2. 王  39023
 3. 張  35457
 4. 陳  26376
 5. 劉  23937
 6. 趙  16548
 7. 楊  15916
 8. 吳  14728
 9. 黃  12650
10. 周  11796
11. 徐  11229
12. 朱  11075
13. 鄭  9782
14. 孫  8301
15. 胡  8191
16. 林  7724
17. 沈  6407
18. 郭  6223
19. 高  6044
20. 何  6030


In [34]:
con = sqlite3.connect("file:" + DB_PATH + "?mode=ro", uri=True)
con.execute("PRAGMA temp_store=MEMORY")
total = con.execute(
    "SELECT COUNT(*) FROM BIOG_MAIN "
    "WHERE c_surname_chn IS NOT NULL AND c_surname_chn <> ''"
).fetchone()[0]
distinct = con.execute(
    "SELECT COUNT(DISTINCT c_personid) FROM BIOG_MAIN "
    "WHERE c_surname_chn IS NOT NULL AND c_surname_chn <> ''"
).fetchone()[0]
con.close()
print("COUNT(*)               :", total)
print("COUNT(DISTINCT personid):", distinct)
print("无重复计数:", total == distinct)

COUNT(*)               : 647915
COUNT(DISTINCT personid): 647915
无重复计数: True
